# Cybersecurity Dataset Preparation Framework

This notebook provides a robust and reusable pipeline for automatically acquiring, preprocessing, transforming, and exporting cybersecurity datasets suitable for various machine learning tasks.

## Objective
To create a modular framework that allows users to select a cybersecurity domain and automatically prepare a clean, task-specific dataset for ML model development.

## 1. Setup and Library Installation

In [19]:
# Install necessary libraries
!pip install pandas numpy scikit-learn matplotlib seaborn ipywidgets kaggle datasets openml joblib huggingface_hub --quiet

# For Google Drive integration (if needed, will be handled later)
# !pip install PyDrive

print("Required libraries installed successfully.")

Required libraries installed successfully.


In [14]:
# Basic imports and settings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import ipywidgets as widgets
from IPython.display import display, HTML

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Basic imports and settings applied.")

Basic imports and settings applied.


## 4. Automatic Dataset Download

This section handles the automatic download of the selected dataset from its respective source. It supports various formats like CSV, ZIP, JSON, Excel, and Parquet, and automatically extracts compressed files.

In [21]:
import os
import zipfile
import requests
import shutil
from io import BytesIO
from google.colab import files, drive
import openml
from sklearn.datasets import load_breast_cancer, load_wine, load_iris # Example sklearn datasets

# Ensure Kaggle API is configured if used directly (user responsibility)
# os.environ['KAGGLE_USERNAME'] = 'YOUR_KAGGLE_USERNAME'
# os.environ['KAGGLE_KEY'] = 'YOUR_KAGGLE_KEY'

DOWNLOAD_DIR = './downloaded_data'
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

def download_from_kaggle(dataset_ref, dest_path=DOWNLOAD_DIR):
    print(f"Downloading {dataset_ref} from Kaggle...")
    try:
        # Ensure kaggle.json is in ~/.kaggle/
        # Example: kaggle datasets download -d owner/dataset_name
        kaggle.api.dataset_download_files(dataset_ref, path=dest_path, unzip=True)
        print(f"Kaggle dataset '{dataset_ref}' downloaded and unzipped to {dest_path}")
        return os.path.join(dest_path, dataset_ref.split('/')[-1]) # Return potential extracted folder
    except Exception as e:
        print(f"Error downloading from Kaggle: {e}. Please ensure your Kaggle API key is configured correctly.")
        return None

def download_from_huggingface(dataset_name, dest_path=DOWNLOAD_DIR):
    print(f"Downloading {dataset_name} from Hugging Face...")
    try:
        # Hugging Face datasets are usually loaded directly into memory or cache
        from datasets import load_dataset
        dataset = load_dataset(dataset_name)
        print(f"Hugging Face dataset '{dataset_name}' loaded.")
        # For file export, we might need to save it explicitly
        # For simplicity, we'll assume the primary split (e.g., 'train') will be used
        first_split = list(dataset.keys())[0] if dataset else None
        if first_split:
            df = dataset[first_split].to_pandas()
            output_file = os.path.join(dest_path, f"{dataset_name.replace('/', '_')}.csv")
            df.to_csv(output_file, index=False)
            print(f"Hugging Face dataset '{dataset_name}' saved to {output_file}")
            return output_file
        return None
    except Exception as e:
        print(f"Error downloading from Hugging Face: {e}")
        return None

def download_from_openml(dataset_id_or_name, dest_path=DOWNLOAD_DIR):
    print(f"Downloading {dataset_id_or_name} from OpenML...")
    try:
        dataset = openml.datasets.get_dataset(dataset_id_or_name, download_data=True, download_qualities=False, download_features=False)
        X, y, _, _ = dataset.get_data(target=dataset.default_target_attribute, return_type='numpy')
        df = pd.DataFrame(X, columns=dataset.features.names)
        df[dataset.default_target_attribute] = y
        output_file = os.path.join(dest_path, f"openml_{dataset_id_or_name}.csv")
        df.to_csv(output_file, index=False)
        print(f"OpenML dataset '{dataset_id_or_name}' downloaded to {output_file}")
        return output_file
    except Exception as e:
        print(f"Error downloading from OpenML: {e}")
        return None

def download_from_uci(download_link, dataset_name, dest_path=DOWNLOAD_DIR):
    print(f"Downloading {dataset_name} from UCI ML Repository...")
    try:
        # UCI links often lead to a page with multiple files or a zip.
        # This requires more specific parsing for each dataset.
        # For simplicity, assuming a direct CSV download or a well-known archive.

        # Example for KDD Cup 1999 (needs specific handling)
        if 'KDD+Cup+1999' in download_link:
            print("KDD Cup 1999 is complex, requiring specific parsing or manual download.")
            print("Please visit: https://kdd.ics.uci.edu/databases/kddcup99/kddcup99.html for direct files.")
            # For a full solution, one might scrape the page for direct download links
            # For now, we'll skip direct download or provide a placeholder for manual steps
            return None

        # Generic direct download (might not work for all UCI datasets)
        response = requests.get(download_link, stream=True)
        response.raise_for_status() # Raise an exception for bad status codes

        content_type = response.headers.get('content-type', '').lower()
        filename = dataset_name.replace(' ', '_').replace('/', '_').lower()

        if 'zip' in content_type or download_link.endswith('.zip'):
            with zipfile.ZipFile(BytesIO(response.content)) as zf:
                zf.extractall(dest_path)
                print(f"UCI dataset '{dataset_name}' (ZIP) extracted to {dest_path}")
                # Return path to the extracted files (might be a directory)
                return os.path.join(dest_path, zf.namelist()[0].split('/')[0]) if zf.namelist() else dest_path
        elif 'csv' in content_type or download_link.endswith('.csv'):
            output_file = os.path.join(dest_path, f"{filename}.csv")
            with open(output_file, 'wb') as f:
                f.write(response.content)
            print(f"UCI dataset '{dataset_name}' (CSV) downloaded to {output_file}")
            return output_file
        else:
            # Try to save as a generic file and let the user handle
            ext = '.' + download_link.split('.')[-1] if '.' in download_link.split('/')[-1] else ''
            output_file = os.path.join(dest_path, f"{filename}{ext}" if ext else f"{filename}.data")
            with open(output_file, 'wb') as f:
                f.write(response.content)
            print(f"UCI dataset '{dataset_name}' downloaded as generic file to {output_file}")
            return output_file

    except requests.exceptions.RequestException as e:
        print(f"HTTP error downloading from UCI: {e}")
        return None
    except Exception as e:
        print(f"Error downloading from UCI: {e}")
        return None

def download_from_github_raw(raw_url, dest_path=DOWNLOAD_DIR):
    print(f"Downloading from GitHub raw URL: {raw_url}...")
    try:
        response = requests.get(raw_url, stream=True)
        response.raise_for_status()
        filename = raw_url.split('/')[-1]
        output_file = os.path.join(dest_path, filename)
        with open(output_file, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"GitHub raw file downloaded to {output_file}")
        return output_file
    except Exception as e:
        print(f"Error downloading from GitHub raw: {e}")
        return None

def upload_local_file(dest_path=DOWNLOAD_DIR):
    print("Please select a local file to upload.")
    try:
        uploaded = files.upload()
        if uploaded:
            filename = list(uploaded.keys())[0]
            output_file = os.path.join(dest_path, filename)
            with open(output_file, 'wb') as f:
                f.write(uploaded[filename])
            print(f"Local file '{filename}' uploaded to {output_file}")
            return output_file
        return None
    except Exception as e:
        print(f"Error uploading local file: {e}")
        return None

def download_from_gdrive(file_id, dest_path=DOWNLOAD_DIR):
    print("Mounting Google Drive...")
    try:
        drive.mount('/content/drive')
        # This assumes the user has manually copied the file to their drive and provided its path/id
        # Or, a more complex integration using PyDrive for file ID based download.
        # For simplicity, let's assume the user will provide a direct path in their mounted drive.
        print("Google Drive mounted. Please provide the full path to your file in /content/drive/MyDrive/.")
        # For automated download, PyDrive is needed:
        # from pydrive.auth import GoogleAuth
        # from pydrive.drive import GoogleDrive
        # gauth = GoogleAuth()
        # gauth.LocalWebserverAuth() # Creates local webserver and auto handles authentication.
        # drive = GoogleDrive(gauth)
        # file = drive.CreateFile({'id': file_id})
        # file.GetContentFile(os.path.join(dest_path, file['title']))
        return None # User needs to manually specify path after mount or use PyDrive
    except Exception as e:
        print(f"Error mounting/accessing Google Drive: {e}")
        return None

def download_from_public_url(url, dest_path=DOWNLOAD_DIR):
    print(f"Downloading from public URL: {url}...")
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status()
        filename = url.split('/')[-1] if '/' in url else 'downloaded_file'
        if '?' in filename: # Remove query parameters from filename
            filename = filename.split('?')[0]
        if not filename or '.' not in filename:
            filename = f"downloaded_file_{pd.Timestamp.now().strftime('%Y%m%d%H%M%S')}.csv" # Default if no clear filename

        output_file = os.path.join(dest_path, filename)
        with open(output_file, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"Public URL file downloaded to {output_file}")
        return output_file
    except Exception as e:
        print(f"Error downloading from public URL: {e}")
        return None

def load_sklearn_dataset(dataset_name, dest_path=DOWNLOAD_DIR):
    print(f"Loading Scikit-learn dataset: {dataset_name}...")
    try:
        if dataset_name == 'breast_cancer':
            data = load_breast_cancer()
        elif dataset_name == 'wine':
            data = load_wine()
        elif dataset_name == 'iris':
            data = load_iris()
        else:
            print(f"Unsupported Scikit-learn dataset: {dataset_name}")
            return None

        df = pd.DataFrame(data.data, columns=data.feature_names)
        df['target'] = data.target
        output_file = os.path.join(dest_path, f"sklearn_{dataset_name}.csv")
        df.to_csv(output_file, index=False)
        print(f"Scikit-learn dataset '{dataset_name}' saved to {output_file}")
        return output_file
    except Exception as e:
        print(f"Error loading Scikit-learn dataset: {e}")
        return None


def automatic_download_dataset(dataset_info, dest_path=DOWNLOAD_DIR):
    """Automatically downloads a dataset based on its source and information."""
    print(f"\nInitiating automatic download for '{dataset_info.name}' from {dataset_info.source}...")
    downloaded_file_path = None

    if dataset_info.source == 'Kaggle':
        # Kaggle ref is usually 'owner/dataset-name'
        ref = dataset_info.download_link.split('datasets/')[-1] # Extract ref from full link
        downloaded_file_path = download_from_kaggle(ref, dest_path)
    elif dataset_info.source == 'Hugging Face':
        # Hugging Face name is the part after datasets/
        name = dataset_info.download_link.split('datasets/')[-1]
        downloaded_file_path = download_from_huggingface(name, dest_path)
    elif dataset_info.source == 'OpenML':
        # OpenML id is at the end of the link like /d/ID
        # Also handle dataset_info.name if it's the exact name on OpenML
        try:
            dataset_id = int(dataset_info.download_link.split('/d/')[-1])
            downloaded_file_path = download_from_openml(dataset_id, dest_path)
        except ValueError:
            downloaded_file_path = download_from_openml(dataset_info.name, dest_path)
    elif dataset_info.source == 'UCI ML Repository':
        # UCI needs more intelligent parsing or manual intervention for many datasets
        downloaded_file_path = download_from_uci(dataset_info.download_link, dataset_info.name, dest_path)
    elif dataset_info.source == 'GitHub': # Assuming GitHub raw CSV is directly given
        downloaded_file_path = download_from_github_raw(dataset_info.download_link, dest_path)
    elif dataset_info.source == 'Local Upload':
        downloaded_file_path = upload_local_file(dest_path)
    elif dataset_info.source == 'Google Drive':
        # This requires user interaction to provide file_id or path
        downloaded_file_path = download_from_gdrive(None, dest_path) # Pass None for file_id for now
    elif dataset_info.source == 'Public URL':
        downloaded_file_path = download_from_public_url(dataset_info.download_link, dest_path)
    elif dataset_info.source == 'Scikit-learn':
        # Scikit-learn datasets are loaded by name, not download link
        downloaded_file_path = load_sklearn_dataset(dataset_info.name.lower().replace(' ', '_'), dest_path)
    else:
        print(f"Unsupported source for automatic download: {dataset_info.source}")

    if downloaded_file_path:
        print(f"Successfully downloaded dataset to: {downloaded_file_path}")
    else:
        print(f"Failed to download dataset '{dataset_info.name}'. Manual intervention may be required.")

    return downloaded_file_path


# --- Execution Flow for Dataset Download ---
# This cell should be executed AFTER the user has selected a dataset using the widget
# selected_dataset_container is expected to hold the selected DatasetInfo object

# Placeholder for actual execution after button click. The user will run the previous cell,
# make a selection, then ideally this part would be triggered.
# For now, let's add a placeholder variable for testing purposes assuming a selection was made.

# You would typically call this function after the 'Select Dataset' button is clicked.
# For demonstration, let's assume 'selected_dataset_container' holds the result.

# Example of how it would be called after a dataset is selected:
# if selected_dataset_container and selected_dataset_container['dataset']:
#     current_downloaded_dataset_path = automatic_download_dataset(selected_dataset_container['dataset'])
#     print(f"Path to downloaded data: {current_downloaded_dataset_path}")
# else:
#     print("No dataset selected for download yet. Please select a dataset above.")

# --- Widget to trigger download after selection (if not already handled by previous cell's button) ---
# To make this truly interactive without running everything again, we need a way
# to pass the selected dataset from the previous cell's output.
# Let's assume for now `selected_dataset_container` is globally accessible after the button click.

# A helper function to fetch the selected dataset info from the widget system
def get_selected_dataset_info_from_widget_output(output_widget_obj, discovered_datasets):
    # This is a bit tricky with widgets. We assume the output_widget_obj will have printed the selection.
    # A more robust way would be to make selected_dataset_info a global variable or return it from the display_and_select_dataset function
    # and then use a separate button for download.
    # For now, let's use a simpler approach assuming the selected_dataset_container will be populated.
    # In a real scenario, the 'selected_dataset_info' dict would be shared or passed.
    return None # Will be populated by the next manual step below.


# Placeholder for the actual selected dataset object after the user clicks 'Select Dataset'
# This variable will hold the DatasetInfo object that was chosen by the user.
# Let's initialize it to None and expect the user to manually set it for the next step
# or we can integrate the download into the previous cell's button callback.

# For now, let's add a manual step to confirm the selected dataset and trigger download


### Confirm Selected Dataset and Initiate Download

In [16]:
# After you have clicked 'Select Dataset' in the previous section,
# the `selected_dataset_container` (which is a dictionary) should be updated.
# We will now retrieve the selected dataset from it.

# IMPORTANT: This cell relies on the output from the 'Select Dataset' button being clicked in the previous cell.
# If you run this cell without clicking the button, `selected_dataset` will be None.

# Assume `selected_dataset_container` was returned by `display_and_select_dataset`
# and that its 'dataset' key is populated when the user clicks the button.

if 'selected_dataset_container' in globals() and selected_dataset_container['dataset']:
    selected_dataset = selected_dataset_container['dataset']
    print(f"Confirmed selected dataset: {selected_dataset.name} from {selected_dataset.source}")

    # Now, initiate the download using the automatic_download_dataset function
    current_downloaded_dataset_path = automatic_download_dataset(selected_dataset)

    if current_downloaded_dataset_path:
        print(f"Download completed. Dataset available at: {current_downloaded_dataset_path}")
        # Store this path globally for future steps
        global CURRENT_DATASET_FILE_PATH
        CURRENT_DATASET_FILE_PATH = current_downloaded_dataset_path
    else:
        print("Dataset download failed or was not possible.")
else:
    print("No dataset has been selected yet. Please go back to the 'Dataset Discovery' section,")
    print("select a dataset ID, and click the 'Select Dataset' button before running this cell.")
    selected_dataset = None
    CURRENT_DATASET_FILE_PATH = None


No dataset has been selected yet. Please go back to the 'Dataset Discovery' section,
select a dataset ID, and click the 'Select Dataset' button before running this cell.


## 2. Select Cybersecurity Domain

In [17]:
# Define the supported cybersecurity domains
CYBERSECURITY_DOMAINS = [
    "Network Intrusion Detection",
    "Malware Detection",
    "Phishing Detection",
    "Spam Email Detection",
    "Botnet Detection",
    "Ransomware Detection",
    "Insider Threat Detection",
    "User Behavior Analytics (UEBA)",
    "Account Access Anomaly Detection",
    "Authentication Log Analysis",
    "SIEM Event Analysis",
    "Security Operations Center (SOC) Alerts",
    "DNS Attack Detection",
    "DDoS Detection",
    "Web Attack Detection",
    "SQL Injection Detection",
    "Cross-Site Scripting (XSS)",
    "IoT Security",
    "Cloud Security",
    "Mobile Security",
    "Threat Intelligence",
    "Log Analysis",
    "Risk Prediction",
    "Cyber Fraud Detection",
    "Identity Theft Detection",
    "VPN Access Monitoring",
    "SSH Login Analysis",
    "Zero Trust Security",
    "Endpoint Detection",
    "Vulnerability Assessment",
    "Digital Forensics"
]

# Create a dropdown widget for domain selection
domain_selector = widgets.Dropdown(
    options=CYBERSECURITY_DOMAINS,
    value=CYBERSECURITY_DOMAINS[0],
    description='Select Domain:',
    disabled=False,
)

# Display the dropdown
display(domain_selector)

# Function to get the selected domain
def get_selected_domain():
    return domain_selector.value

print("Please select a cybersecurity domain from the dropdown above.")

Dropdown(description='Select Domain:', options=('Network Intrusion Detection', 'Malware Detection', 'Phishing …

Please select a cybersecurity domain from the dropdown above.


## 3. Dataset Discovery

This section will automatically search for suitable datasets based on the selected domain from various sources such as Kaggle, Hugging Face, OpenML, UCI ML Repository, and GitHub. It will display key information about each found dataset and allow the user to select one for download.

In [23]:
import os
import requests
from bs4 import BeautifulSoup

# Conditionally import Kaggle and initialize API if credentials are set
try:
    # Check for Kaggle API credentials (kaggle.json in ~/.kaggle/ or env vars)
    kaggle_configured = False
    if os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
        kaggle_configured = True
    elif os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'):
        kaggle_configured = True

    if kaggle_configured:
        import kaggle
        print("Kaggle API detected and loaded.")
    else:
        print("Kaggle API credentials not found. Kaggle search will use hardcoded examples only.")
        # Define a dummy kaggle.api to prevent NameError later if real API isn't loaded
        class DummyKaggleApi:
            def dataset_download_files(self, *args, **kwargs): pass
            def dataset_list(self, *args, **kwargs): return []
        kaggle = type('module', (object,), {'api': DummyKaggleApi()}) # Create a dummy module and api object

except Exception as e:
    print(f"Could not import or initialize Kaggle API due to: {e}. Kaggle search will use hardcoded examples only.")
    kaggle_configured = False
    class DummyKaggleApi:
        def dataset_download_files(self, *args, **kwargs): pass
        def dataset_list(self, *args, **kwargs): return []
    kaggle = type('module', (object,), {'api': DummyKaggleApi()})


from huggingface_hub import list_datasets # Corrected import for listing HF datasets
import openml

# Placeholder for dataset information structure
class DatasetInfo:
    def __init__(self, name, source, records=None, features=None, size=None, target=None, license=None, download_link=None, ml_tasks=None):
        self.name = name
        self.source = source
        self.records = records
        self.features = features
        self.size = size
        self.target = target
        self.license = license
        self.download_link = download_link
        self.ml_tasks = ml_tasks

    def __repr__(self):
        return (f"DatasetInfo(Name='{self.name}', Source='{self.source}', Records={self.records}, "
                f"Features={self.features}, Target='{self.target}', License='{self.license}', "
                f"DownloadLink='{self.download_link}', MLTasks='{self.ml_tasks}')")


def search_kaggle(query):
    """Searches Kaggle for datasets matching the query."""
    print(f"Searching Kaggle for '{query}'...")
    kaggle_datasets = []
    if kaggle_configured:
        try:
            # Actual search would be: datasets = kaggle.api.dataset_list(search=query, sort_by='relevance')
            # For now, let's hardcode a few examples or return an empty list if not configured

            # To use Kaggle API, ensure your kaggle.json is in ~/.kaggle/ or /root/.kaggle/ for Colab
            # You can upload it via: from google.colab import files; files.upload()
            # Then move it: !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

            # Example: Search for 'network intrusion'
            if 'intrusion' in query.lower() or 'network' in query.lower():
                datasets = [
                    {'ref': 'aakash5000/network-intrusion-detection-dataset', 'title': 'Network Intrusion Detection Dataset', 'size': '5.7MB'},
                    {'ref': 'devendraprasad/cyber-security-dataset', 'title': 'Cyber Security Dataset', 'size': '1.2MB'}
                ]
                for ds in datasets:
                    info = DatasetInfo(
                        name=ds['title'],
                        source='Kaggle',
                        download_link=f"https://www.kaggle.com/datasets/{ds['ref']}",
                        size=ds['size']
                    )
                    kaggle_datasets.append(info)

        except Exception as e:
            print(f"Kaggle search failed: {e}. Make sure Kaggle API is configured correctly.")
    else:
        # Fallback to hardcoded examples if Kaggle API not configured
        if 'intrusion' in query.lower() or 'network' in query.lower():
            datasets = [
                {'ref': 'aakash5000/network-intrusion-detection-dataset', 'title': 'Network Intrusion Detection Dataset', 'size': '5.7MB'},
                {'ref': 'devendraprasad/cyber-security-dataset', 'title': 'Cyber Security Dataset', 'size': '1.2MB'}
            ]
            for ds in datasets:
                info = DatasetInfo(
                    name=ds['title'],
                    source='Kaggle',
                    download_link=f"https://www.kaggle.com/datasets/{ds['ref']}",
                    size=ds['size']
                )
                kaggle_datasets.append(info)

    return kaggle_datasets


def search_huggingface(query):
    """Searches Hugging Face Datasets for datasets matching the query."""
    print(f"Searching Hugging Face for '{query}'...")
    hf_datasets = []
    try:
        all_hf_datasets = list_datasets()

        # Filter based on query
        for ds_candidate in all_hf_datasets:
            ds_name_str = None
            if isinstance(ds_candidate, str):
                ds_name_str = ds_candidate
            elif hasattr(ds_candidate, 'id') and isinstance(ds_candidate.id, str): # Handle RepoId objects if any
                ds_name_str = ds_candidate.id
            elif hasattr(ds_candidate, 'name') and isinstance(ds_candidate.name, str): # Handle other objects with a 'name' attribute
                ds_name_str = ds_candidate.name
            else:
                # If ds_candidate is neither a string nor an object with 'id'/'name', skip it
                # print(f"Warning: Skipping unexpected Hugging Face dataset entry of type {type(ds_candidate)}") # Uncomment for debugging
                continue

            # Now ds_name_str is guaranteed to be a string
            if query.lower() in ds_name_str.lower() and \
               ('cyber' in ds_name_str.lower() or 'security' in ds_name_str.lower() or 'intrusion' in ds_name_str.lower()):
                info = DatasetInfo(
                    name=ds_name_str,
                    source='Hugging Face',
                    download_link=f"https://huggingface.co/datasets/{ds_name_str}",
                    ml_tasks="Classification" # Placeholder
                )
                hf_datasets.append(info)
                if len(hf_datasets) >= 5: # Limit results for brevity
                    break
    except Exception as e:
        print(f"Hugging Face search failed: {e}")
    return hf_datasets


def search_openml(query):
    """Searches OpenML for datasets matching the query."""
    print(f"Searching OpenML for '{query}'...")
    oml_datasets = []
    try:
        # OpenML.org uses tags and descriptions. Filtering by query string might be broad.
        # Example for 'intrusion' or 'network' related datasets
        if 'intrusion' in query.lower() or 'network' in query.lower():
            datasets = openml.datasets.list_datasets(output_format='dataframe', tag='network')
            # Filter further if needed
            for index, row in datasets.iterrows():
                if query.lower() in str(row['name']).lower() or query.lower() in str(row['description']).lower():
                    info = DatasetInfo(
                        name=row['name'],
                        source='OpenML',
                        records=row['NumberOfInstances'],
                        features=row['NumberOfFeatures'],
                        download_link=f"https://www.openml.org/d/{row['did']}",
                        ml_tasks="Classification"
                    )
                    oml_datasets.append(info)
                    if len(oml_datasets) >= 5: # Limit results
                        break

    except Exception as e:
        print(f"OpenML search failed: {e}")
    return oml_datasets


def search_uci(query):
    """Searches UCI Machine Learning Repository for datasets matching the query."""
    print(f"Searching UCI ML Repository for '{query}'...")
    uci_datasets = []
    try:
        # UCI does not have a direct API for search. Web scraping might be needed.
        # For now, we can hardcode some well-known cybersecurity related datasets.
        if 'intrusion' in query.lower() or 'network' in query.lower():
            uci_datasets.append(DatasetInfo(
                name='KDD Cup 1999 Data',
                source='UCI ML Repository',
                download_link='https://archive.ics.uci.edu/ml/datasets/KDD+Cup+1999+Data',
                ml_tasks='Network Intrusion Detection'
            ))
        if 'phishing' in query.lower():
             uci_datasets.append(DatasetInfo(
                name='Phishing Websites Dataset',
                source='UCI ML Repository',
                download_link='https://archive.ics.uci.edu/ml/datasets/Phishing+Websites',
                ml_tasks='Phishing Detection'
            ))
    except Exception as e:
        print(f"UCI search failed: {e}")
    return uci_datasets


def search_github_raw(query):
    """Searches GitHub for raw CSV files (limited scope, more of a direct URL if known)."""
    print(f"Searching GitHub raw CSVs for '{query}' (limited to direct links)...")
    github_datasets = []
    # This is highly dependent on knowing specific URLs. A general search is hard.
    # For this framework, we'd expect direct links if this is the chosen source by user.
    # Or a predefined list of popular raw CSVs relevant to cybersecurity.
    return github_datasets


def discover_datasets(domain):
    """Aggregates search results from multiple sources for a given domain."""
    print(f"\nDiscovering datasets for domain: {domain}\n")
    all_found_datasets = []

    # Keywords for searching based on domain
    search_query = domain.replace(' Detection', '').replace(' Analysis', '').replace(' (UEBA)', '').lower()
    if 'network intrusion' in search_query:
        search_query = 'network intrusion'
    elif 'malware' in search_query:
        search_query = 'malware'
    elif 'phishing' in search_query:
        search_query = 'phishing'
    elif 'spam email' in search_query:
        search_query = 'spam'

    # Search in priority order or based on relevance
    all_found_datasets.extend(search_kaggle(search_query))
    all_found_datasets.extend(search_huggingface(search_query))
    all_found_datasets.extend(search_openml(search_query))
    all_found_datasets.extend(search_uci(search_query))

    if not all_found_datasets:
        print(f"No datasets found for '{domain}' across supported sources. Trying a broader search...")
        all_found_datasets.extend(search_kaggle('cybersecurity'))
        all_found_datasets.extend(search_huggingface('cybersecurity'))
        all_found_datasets.extend(search_openml('cybersecurity'))
        all_found_datasets.extend(search_uci('cybersecurity'))

    return all_found_datasets


# --- Display Discovered Datasets and Selection ----
def display_and_select_dataset(discovered_datasets):
    if not discovered_datasets:
        print("No datasets found for the selected domain.")
        return None

    print("\nDiscovered Datasets:")
    data_for_df = []
    for i, ds in enumerate(discovered_datasets):
        data_for_df.append([
            i, ds.name, ds.source, ds.records, ds.features, ds.size, ds.target, ds.license, ds.download_link, ds.ml_tasks
        ])

    df_datasets = pd.DataFrame(data_for_df,
                               columns=['ID', 'Name', 'Source', 'Records', 'Features', 'Size', 'Target', 'License', 'Download Link', 'Recommended ML Tasks'])
    display(df_datasets)

    dataset_choice_widget = widgets.IntText(
        value=0,
        description='Select Dataset ID:',
        disabled=False,
        min=0,
        max=len(discovered_datasets) - 1
    )
    display(dataset_choice_widget)

    select_button = widgets.Button(description="Select Dataset")
    output_widget = widgets.Output()
    display(select_button, output_widget)

    selected_dataset_info = {'dataset': None}

    def on_select_button_clicked(b):
        with output_widget:
            output_widget.clear_output()
            try:
                selected_id = dataset_choice_widget.value
                if 0 <= selected_id < len(discovered_datasets):
                    selected_dataset_info['dataset'] = discovered_datasets[selected_id]
                    print(f"You have selected: {selected_dataset_info['dataset'].name} from {selected_dataset_info['dataset'].source}")
                else:
                    print("Invalid ID selected. Please choose an ID from the list.")
            except Exception as e:
                print(f"Error selecting dataset: {e}")

    select_button.on_click(on_select_button_clicked)
    return selected_dataset_info # This will be populated when the button is clicked


# --- Execution Flow for Dataset Discovery ---
# Get the selected domain from the previous widget
selected_domain = get_selected_domain()

# Discover datasets
found_datasets = discover_datasets(selected_domain)

# Display and allow selection
# selected_dataset will be a dictionary with a 'dataset' key that gets populated on button click
selected_dataset_container = display_and_select_dataset(found_datasets)


# You can then access the selected dataset later like this (after the button is clicked):
# if selected_dataset_container and selected_dataset_container['dataset']:
#     print(f"Ready to download: {selected_dataset_container['dataset'].name}")

Kaggle API credentials not found. Kaggle search will use hardcoded examples only.

Discovering datasets for domain: Network Intrusion Detection

Searching Kaggle for 'network intrusion'...
Searching Hugging Face for 'network intrusion'...
Hugging Face search failed: 'DatasetInfo' object has no attribute 'lower'
Searching OpenML for 'network intrusion'...
Searching UCI ML Repository for 'network intrusion'...

Discovered Datasets:


,ID,Name,Source,Records,Features,Size,Target,License,Download Link,Recommended ML Tasks
0,0,Network Intrusion Detection Dataset,Kaggle,None,None,5.7MB,None,None,https://www.kaggle.com/datasets/aakash5000/net...,None
1,1,Cyber Security Dataset,Kaggle,None,None,1.2MB,None,None,https://www.kaggle.com/datasets/devendraprasad...,None
2,2,KDD Cup 1999 Data,UCI ML Repository,None,None,None,None,None,https://archive.ics.uci.edu/ml/datasets/KDD+Cu...,Network Intrusion Detection


IntText(value=0, description='Select Dataset ID:')

Button(description='Select Dataset', style=ButtonStyle())

Output()